In [3]:
import pandas as pd

# =====================================================
# Load the 3 scoring panels
# =====================================================

hvg_df = pd.read_csv("HVG_panel.csv")[["Ensembl_ID", "HVG_score"]]
de_df  = pd.read_csv("DE_panel_leiden_0.6.csv")[["Ensembl_ID", "DE_global_score"]]
pca_df = pd.read_csv("PCA_genes_full_panel.csv")[["Ensembl_ID", "PCA_combined_score"]]

# =====================================================
# Merge on Ensembl_ID
# =====================================================

merged = (
    hvg_df
    .merge(de_df, on="Ensembl_ID")
    .merge(pca_df, on="Ensembl_ID")
)

print("Merged shape:", merged.shape)


# =====================================================
# NORMALIZE INDIVIDUAL SCORES (min-max)
# =====================================================

def minmax_norm(x):
    return (x - x.min()) / (x.max() - x.min())

merged["HVG_norm"] = minmax_norm(merged["HVG_score"])
merged["DE_norm"]  = minmax_norm(merged["DE_global_score"])
merged["PCA_norm"] = minmax_norm(merged["PCA_combined_score"])


# =====================================================
# Combine normalized scores
# =====================================================

merged["Combined_score"] = (
    merged["HVG_norm"] +
    merged["DE_norm"] +
    merged["PCA_norm"]
)

merged["Combined_score"] = merged["Combined_score"].clip(lower=0)


# =====================================================
# Convert combined score into REAL probabilities
# =====================================================

total = merged["Combined_score"].sum()
merged["Combined_score_prob"] = merged["Combined_score"] / total

print("Sum of all probabilities:", merged["Combined_score_prob"].sum())


# =====================================================
# ORDER GENES BY PROBABILITY (MOST LIKELY FIRST)
# =====================================================

merged_sorted = merged.sort_values("Combined_score_prob", ascending=False)

# =====================================================
# Save results
# =====================================================

# Full file: all columns + ordered
merged_sorted.to_csv("combined_panel_full_sorted.csv", index=False)

# Minimal file: ID + probability, ordered
merged_sorted[["Ensembl_ID", "Combined_score_prob"]].to_csv(
    "combined_panel_sorted.csv",
    index=False
)

print("\nSaved:")
print("- combined_panel_full_sorted.csv (ALL columns, ordered)")
print("- combined_panel_sorted.csv (ID + probability, ordered)")

print("\nTOP 5 GENES (by probability):")
print(merged_sorted.head())


Merged shape: (30208, 4)
Sum of all probabilities: 1.0

Saved:
- combined_panel_full_sorted.csv (ALL columns, ordered)
- combined_panel_sorted.csv (ID + probability, ordered)

TOP 5 GENES (by probability):
            Ensembl_ID  HVG_score  DE_global_score  PCA_combined_score  \
52  ENSMUSG00000033740   3.776116        232.49530            0.003610   
99  ENSMUSG00000048960   3.389156        225.40349            0.003491   
0   ENSMUSG00000007097   6.043343         74.07136            0.004176   
90  ENSMUSG00000062209   3.529990        152.66006            0.003483   
1   ENSMUSG00000026473   6.004214         82.66842            0.003445   

    HVG_norm   DE_norm  PCA_norm  Combined_score  Combined_score_prob  
52  0.769152  0.968128  0.864543        2.601822             0.000131  
99  0.729752  0.941120  0.836043        2.506914             0.000126  
0   1.000000  0.364790  1.000000        2.364790             0.000119  
90  0.744092  0.664085  0.834057        2.242234             

In [4]:
import pandas as pd
import numpy as np

# =====================================================
# USER CONFIGURATION
# =====================================================

# Probability distribution mode:
# "linear"    → 1 / rank_score
# "log"       → log1p(1 / rank_score)        (recommended)
# "log_soft"  → log1p(1 / sqrt(rank_score))
# "log_flat"  → 1 / log1p(rank_score + 1)
prob_mode = "log"   # <-- CHANGE HERE

# Lexicographic weights (min rank dominates)
weights = [1.0, 0.01, 0.0001]

# Numerical stability
EPS = 1e-9

# =====================================================
# LOAD PANELS
# =====================================================

hvg_df = pd.read_csv("HVG_panel.csv")[["Ensembl_ID", "HVG_score"]]
de_df  = pd.read_csv("DE_panel_leiden_0.6.csv")[["Ensembl_ID", "DE_global_score"]]
pca_df = pd.read_csv("PCA_genes_full_panel.csv")[["Ensembl_ID", "PCA_combined_score"]]

merged = (
    hvg_df
    .merge(de_df, on="Ensembl_ID")
    .merge(pca_df, on="Ensembl_ID")
)

print("Merged shape:", merged.shape)

# =====================================================
# SCORES → RANKS (rank basso = meglio)
# =====================================================

merged["HVG_rank"] = merged["HVG_score"].rank(
    ascending=False, method="min"
)
merged["DE_rank"] = merged["DE_global_score"].rank(
    ascending=False, method="min"
)
merged["PCA_rank"] = merged["PCA_combined_score"].rank(
    ascending=False, method="min"
)

# =====================================================
# RANK PROFILE (min dominates)
# =====================================================

rank_cols = ["HVG_rank", "DE_rank", "PCA_rank"]

merged["rank_profile"] = merged[rank_cols].apply(
    lambda r: tuple(sorted(r)), axis=1
)

# =====================================================
# LEXICOGRAPHIC RANK SCORE (smaller = better)
# =====================================================

merged["rank_score"] = merged["rank_profile"].apply(
    lambda r: sum(w * v for w, v in zip(weights, r))
)

# =====================================================
# RANK SCORE → PROBABILITY (FLAG-BASED)
# =====================================================

if prob_mode == "linear":
    merged["prob_raw"] = 1 / (merged["rank_score"] + EPS)

elif prob_mode == "log":
    merged["prob_raw"] = np.log1p(1 / (merged["rank_score"] + EPS))

elif prob_mode == "log_soft":
    merged["prob_raw"] = np.log1p(1 / np.sqrt(merged["rank_score"] + EPS))

elif prob_mode == "log_flat":
    merged["prob_raw"] = 1 / np.log1p(merged["rank_score"] + 1)

else:
    raise ValueError(f"Unknown prob_mode: {prob_mode}")

# =====================================================
# NORMALIZE → TRUE PROBABILITY DISTRIBUTION
# =====================================================

merged["Combined_score_prob"] = (
    merged["prob_raw"] / merged["prob_raw"].sum()
)

print("Sum of probabilities:", merged["Combined_score_prob"].sum())

# =====================================================
# ORDER GENES (CONSISTENT WITH RANKING)
# =====================================================

merged_sorted = merged.sort_values(
    by="rank_score",
    ascending=True
)

# =====================================================
# SAVE RESULTS
# =====================================================

merged_sorted.to_csv(
    "combined_panel_full_sorted_max.csv",
    index=False
)

merged_sorted[["Ensembl_ID", "Combined_score_prob"]].to_csv(
    "combined_panel_sorted_max.csv",
    index=False
)

print("\nSaved:")
print("- combined_panel_full_sorted.csv (ALL columns, ordered)")
print("- combined_panel_sorted.csv (ID + probability, ordered)")

print("\nTOP 5 GENES:")
print(merged_sorted.head())


Merged shape: (30208, 4)
Sum of probabilities: 1.0000000000000002

Saved:
- combined_panel_full_sorted.csv (ALL columns, ordered)
- combined_panel_sorted.csv (ID + probability, ordered)

TOP 5 GENES:
            Ensembl_ID  HVG_score  DE_global_score  PCA_combined_score  \
0   ENSMUSG00000007097   6.043343        74.071360            0.004176   
1   ENSMUSG00000026473   6.004214        82.668420            0.003445   
12  ENSMUSG00000046240   4.788110         0.290675            0.004076   
2   ENSMUSG00000028565   5.750812        23.818035            0.003566   
58  ENSMUSG00000105265   3.762106        37.760014            0.004009   

    HVG_rank  DE_rank  PCA_rank          rank_profile  rank_score  prob_raw  \
0        1.0   1496.0       1.0    (1.0, 1.0, 1496.0)      1.1596  0.621848   
1        2.0   1235.0      54.0   (2.0, 54.0, 1235.0)      2.6635  0.318778   
12      13.0  14987.0       2.0  (2.0, 13.0, 14987.0)      3.6287  0.243402   
2        3.0   6240.0      40.0   (3.0,